# Earthquake Detection

In [1]:
import sys
# Self-contained: the helper modules (comcat_check, plot_pick_check, plot_das_bb_check,
# detect_utilSEISMIC_fromQibin) sit next to this notebook and the small inputs are in ./data/.
# Run with the test_environment conda kernel (ELEP, pyocto, seisbench, torch, cartopy).

import gc
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seisbench.models as sbm

import torch
import pyocto
from ELEP.elep.ensemble_coherence import ensemble_semblance
from ELEP.elep.trigger_func import picks_summary_simple

from datetime import datetime
from detect_utilSEISMIC_fromQibin import apply_elep_smb, detect_on_fly, detect_on_fly2, detect_on_fly3  # local copy of DAS-denoise-detect-classify/src/models/detect_utilSEISMIC_fromQibin.py
from joblib import Parallel, delayed
from tqdm import tqdm

import obspy
from obspy import UTCDateTime, read_events, read, read_inventory
from obspy.clients.fdsn.client import Client

###
#filepath = '/fd1/QibinShi_data/akdas/qibin_data/elep_pyocto/coast_only/'
filepath = "/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/"
out_path = "/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/association/"

# Association

In [2]:
# Window comes from the DAS campaign index, not a hardcoded pair of dates. These are the real
# first and last files on disk, so the broadband request cannot drift from the DAS coverage it
# will be compared against, and it re-derives itself if the campaign is ever re-indexed.
FILE_INDEX = ('/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/phasenet-3-month-run_08182026/file_index.csv')
DAS_FILE_SECONDS = 60      # each archive file holds 60 s, so the last file ENDS 60 s after it starts

_idx = pd.read_csv(FILE_INDEX, usecols=['file_time'], parse_dates=['file_time'])

starttime1 = UTCDateTime(_idx.file_time.min().isoformat())
endtime1   = UTCDateTime(_idx.file_time.max().isoformat()) + DAS_FILE_SECONDS

print('DAS index : %d files over %d days' % (len(_idx), _idx.file_time.dt.date.nunique()))
print('starttime1: %s   (first DAS file)' % starttime1)
print('endtime1  : %s   (last DAS file + %d s)' % (endtime1, DAS_FILE_SECONDS))
print('span      : %.2f days' % ((endtime1 - starttime1) / 86400))

# Coverage is not continuous inside that span -- 2023-11-09 has no data at all, and six days
# are partial. Request the whole window, but expect no DAS to associate against on those hours.
_per_day = _idx.file_time.dt.date.value_counts()
print('\npartial days (< 1440 files):')
print(_per_day[_per_day < 1440].sort_index().to_string())

DAS index : 126458 files over 90 days
starttime1: 2023-08-23T19:19:28.000000Z   (first DAS file)
endtime1  : 2023-11-22T00:00:00.000000Z   (last DAS file + 60 s)
span      : 90.19 days

partial days (< 1440 files):
file_time
2023-08-23     281
2023-09-18     471
2023-09-19     664
2023-10-14    1330
2023-11-08    1371
2023-11-10    1380


In [3]:
# Station list for the get_stations() call below, read from the FDSN text file rather than
# retyped into a string -- so adding a station to the file is the only edit needed.
STATION_TXT = 'data/MtRainier_seis.txt'   # '#Network|Station|Lat|Lon|Elev|Sitename|StartTime|EndTime'

# Keep only stations actually operating during the DAS window from the cell above. The file
# carries 1995 NEHRP temporary deployments and stations decommissioned years before this
# campaign (RVC ended 2021-01-10, WPW 2017-04-19); requesting them is silently empty, and they
# would otherwise look like stations that simply produced no picks.
FILTER_TO_DAS_WINDOW = True

_sta = pd.read_csv(STATION_TXT, sep='|')
_sta.columns = [c.lstrip('#').strip() for c in _sta.columns]
_sta['StartTime'] = pd.to_datetime(_sta['StartTime'], errors='coerce')
_sta['EndTime']   = pd.to_datetime(_sta['EndTime'],   errors='coerce')

if FILTER_TO_DAS_WINDOW:
    _t1, _t2 = pd.Timestamp(starttime1.datetime), pd.Timestamp(endtime1.datetime)
    _live = _sta[(_sta['StartTime'] <= _t2) & (_sta['EndTime'] >= _t1)]
else:
    _live = _sta

network = ','.join(sorted(_live['Network'].unique()))
station = ','.join(sorted(_live['Station'].unique()))

print('%s: %d stations, %d live in the DAS window' % (STATION_TXT, len(_sta), len(_live)))
print(_live.groupby('Network')['Station'].count().to_string())
print('\nnetwork = %r' % network)
print('station = %r' % station)

_dropped = _sta[~_sta.index.isin(_live.index)]
if len(_dropped):
    print('\ndropped (not operating in the window):')
    print(_dropped[['Network', 'Station', 'StartTime', 'EndTime']].to_string(index=False))

data/MtRainier_seis.txt: 48 stations, 33 live in the DAS window
Network
CC    25
UW     8

network = 'CC,UW'
station = 'ARAT,CARB,COPP,CRBN,FMW,GNOB,GOBB,GTWY,KAUT,LO2,LON,MILD,OBSR,OPCH,PANH,PARA,PR01,PR02,PR03,PR04,PR05,RCM,RCS,RER,RUSH,SIFT,STAR,STYX,TABR,TAVI,VOIT,WOW,WPEQ'

dropped (not operating in the window):
Network Station  StartTime             EndTime
     CC    SR41 2015-07-09 2018-04-28 00:00:00
     UW     ASF 1995-03-20 1995-06-12 23:59:59
     UW     DLR 1995-04-07 1995-09-19 23:59:59
     UW     FGM 1995-05-20 1995-09-19 23:59:59
     UW     GRL 1995-10-01 1996-07-18 23:59:59
     UW     MHL 1995-08-02 1995-10-31 23:59:59
     UW     RSH 1995-06-13 1995-09-20 23:59:59
     UW     RSU 1999-08-20 2001-09-21 00:00:00
     UW     RVC 1983-01-01 2021-01-10 00:00:00
     UW     RVN 1995-09-26 2002-08-15 00:00:00
     UW     WCR 1995-06-12 1995-09-22 23:59:59
     UW     WI2 1995-09-24 1996-07-17 23:59:59
     UW     WPS 1995-06-29 1995-09-29 23:59:59
     UW     WPW 1980-03

In [4]:
# Station inventories for the PyOcto station table below. The picks in filepath were made
# with these same two inventories (BH? via detect_on_fly, HH? via detect_on_fly2/3), so
# the table built from them matches the station codes in picks_octo_sta.csv.
client = Client("IRIS")

t1 = starttime1
t2 = endtime1

inventory = client.get_stations(network=network, station = station, channel="BH?",
                                starttime=t1, endtime=t2)

inventory2 = client.get_stations(network=network, station = station, channel="HH?",
                                 starttime=t1, endtime=t2)

In [5]:
t1,t2

(2023-08-23T19:19:28.000000Z, 2023-11-22T00:00:00.000000Z)

In [6]:
inventory

Inventory created at 2026-11-03T22:36:45.131900Z
	Created by: EarthScope WEB SERVICE: fdsnws-station | version: 1.1.59
		    https://service.earthscope.org/fdsnws/station/1/query?starttime=202...
	Sending institution: EarthScope (EarthScope)
	Contains:
		Networks (1):
			CC
		Stations (24):
			CC.ARAT (Mount Ararat)
			CC.CARB (Carbon Repeater)
			CC.COPP (Copper Mountain)
			CC.CRBN (Carbon River Ranger Station)
			CC.GNOB (Gobblers Knob Seismic)
			CC.GTWY (Gateway Entrance Station)
			CC.KAUT (Kautz Creek Helibase)
			CC.MILD (Mildred Point)
			CC.OBSR (Observation Rock)
			CC.OPCH (Ohanapecosh Administrative Area)
			CC.PANH (Panhandle Gap)
			CC.PARA (Paradise Precip Tower)
			CC.PR01 (Puyallup River 01)
			CC.PR02 (Puyallup River 02)
			CC.PR03 (Puyallup River 03)
			CC.PR04 (Puyallup River 04)
			CC.PR05 (Puyallup River 05)
			CC.RUSH (Rushingwater Creek)
			CC.SIFT (Swift Creek)
			CC.STYX (STYX, Puyallup River, WA, USA)
			CC.TABR (Tahoma Bridge)
			CC.TAVI (Tahoma Vista)
			C

In [7]:
inventory2

Inventory created at 2026-11-03T22:36:45.579100Z
	Created by: EarthScope WEB SERVICE: fdsnws-station | version: 1.1.59
		    https://service.earthscope.org/fdsnws/station/1/query?starttime=202...
	Sending institution: EarthScope (EarthScope)
	Contains:
		Networks (1):
			UW
		Stations (5):
			UW.FMW (Mt. Fremont, WA, USA)
			UW.LON (Longmire, WA, USA)
			UW.RCM (Camp Muir, Mount Rainier, WA, USA)
			UW.RER (Emerald Ridge, Mount Rainier, WA, USA)
			UW.WPEQ (White Pass Ski Facility, WA, USA)
		Channels (0):


In [8]:
### Merge picks from all stations
csv_list = glob.glob(filepath+'/*2023*.csv')
all_csv =[]
for i in csv_list:
    all_csv.append(pd.read_csv(i, index_col=0))
df = pd.concat(all_csv, axis=0)

# change to Octo format
df.loc[df['trace_p_arrival'].notna(), 'phase'] = "P"
sta_p_time = df.loc[df['trace_p_arrival'].notna(), ['station_code','phase','trace_p_arrival']].rename(
    columns={"station_code": "station", "trace_p_arrival": "time"})

df.loc[df['trace_s_arrival'].notna(), 'phase'] = "S"
sta_s_time = df.loc[df['trace_s_arrival'].notna(), ['station_code','phase','trace_s_arrival']].rename(
    columns={"station_code": "station", "trace_s_arrival": "time"})

picks = pd.concat(objs = [sta_p_time,sta_s_time] , axis=0)
picks['time'] = picks['time'].apply(lambda x: UTCDateTime(x).timestamp)

picks.to_csv(filepath + 'picks_octo_sta.csv', index=False)

In [9]:
layers = pd.read_csv("data/vmodel/IASP91_1.csv").iloc[:78]
print(layers.dtypes)
print(layers.isna().sum())
print(layers.head())
print(layers.tail())
import os
if not os.path.exists("data/vmodel/velocity_model"):
    # The prebuilt table ships in data/vmodel/. Rebuilding needs pyrocko and currently fails
    # under numpy 2.4 (test_environment); only run this if the table is missing.
    pyocto.VelocityModel1D.create_model(layers, 1., 600, 400, "data/vmodel/velocity_model")

## Station table for PyOcto -- the UNION of the BH? and HH? inventories.
# The pick merge above globs every *2023*.csv in filepath, so the picks carry stations from
# BOTH the detect_on_fly (BH?) and detect_on_fly2 (HH?) runs. Built from `inventory` alone
# this table drops every HH?-only station, and PyOcto silently ignores picks whose station id
# is missing from the table -- those stations just never associate, with no warning.
#
# Dedupe on sta.code, which is the key the picks use (station_code, no network prefix; no
# code collides across networks in MtRainier_seis.txt). `inventory + inventory2` does NOT do
# this: obspy's Inventory.__iadd__ only concatenates the network lists, so UW would show up
# twice. A station running both BH and HH is in both inventories, and get_stations() returns
# one Station object per operating epoch, so the duplicates have to be collapsed either way.
#
# Not `station = sta.code` as before: that name holds the comma-separated request string
# built from MtRainier_seis.txt, and rebinding it here would leave it pointing at whichever
# station the loop happened to end on.
rows = {}
for chan, inv in (('BH?', inventory), ('HH?', inventory2)):
    n0 = len(rows)
    for net in inv:
        for sta in net:
            rows.setdefault(sta.code, {'id': sta.code,
                                       'longitude': sta.longitude,
                                       'latitude': sta.latitude,
                                       'elevation': sta.elevation})
    print('%s: %d stations new to the table' % (chan, len(rows) - n0))

stations_table = pd.DataFrame(list(rows.values()),
                              columns=['id', 'longitude', 'latitude', 'elevation'])
print('%d unique stations total' % len(stations_table))
# stations_table = pd.read_csv(filepath + 'stations_table.csv')
# picks_octo_sta.csv, not picks_octo.csv -- that is the name the merge cell above writes,
# and no picks_octo.csv exists in filepath, so the old read was a FileNotFoundError.
picks = pd.read_csv(filepath + 'picks_octo_sta.csv')

velocity_model = pyocto.VelocityModel1D("data/vmodel/velocity_model", tolerance=2.0)

associator = pyocto.OctoAssociator.from_area(
    lat=(45, 48),
    lon=(-123, -120),
    zlim=(0, 40),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=10,
    n_p_picks=3,
    n_s_picks=3,
    n_p_and_s_picks=3,
)
    
# Adds the x/y/z columns (km, in the associator's local frame) that associate() requires.
# This used to be safe to skip only because stations_table was loaded from the cached
# stations_table.csv, which was already transformed; the table is rebuilt from the two
# inventories on every run now, so it arrives with lon/lat/elev only.
associator.transform_stations(stations_table)
# stations_table.to_csv(filepath + 'stations_table.csv', index=False)
stations_table.head(), picks.head()


depth    float64
r        float64
vp       float64
vs       float64
dtype: object
depth    0
r        0
vp       0
vs       0
dtype: int64
   depth       r   vp    vs
0    0.0  6371.0  5.8  3.36
1    1.0  6370.0  5.8  3.36
2    2.0  6369.0  5.8  3.36
3    3.0  6368.0  5.8  3.36
4    4.0  6367.0  5.8  3.36
    depth       r      vp      vs
73  360.0  6011.0  8.8475  4.7830
74  370.0  6001.0  8.8840  4.8004
75  380.0  5991.0  8.9205  4.8178
76  390.0  5981.0  8.9570  4.8352
77  400.0  5971.0  8.9935  4.8526
BH?: 24 stations new to the table
HH?: 5 stations new to the table
29 unique stations total


(     id   longitude   latitude  elevation          x          y        z
 0  ARAT -121.853000  46.789000    1822.44 -26.953268  32.186884 -1.82244
 1  CARB -122.005410  46.988320     872.47 -38.447804  54.408505 -0.87247
 2  COPP -121.828750  46.797980    1895.00 -25.097489  33.177132 -1.89500
 3  CRBN -121.961026  46.988155     500.08 -35.071515  54.369340 -0.50008
 4  GNOB -121.914380  46.794140    1659.36 -31.636913  32.781155 -1.65936,
   station phase          time
 0    ARAT     P  1.692823e+09
 1    COPP     P  1.692821e+09
 2    COPP     P  1.692823e+09
 3    COPP     P  1.692825e+09
 4    COPP     P  1.692829e+09)

In [10]:
# Association 
# Re-project the stations here rather than relying on the table cell having been run in this
# kernel: associate() hard-errors if x/y/z are missing, and the table is rebuilt from the two
# inventories on every run so it arrives with lon/lat/elev only. transform_stations is
# idempotent -- it recomputes x/y/z from latitude/longitude/elevation -- so calling it in both
# places costs nothing.
associator.transform_stations(stations_table)
print('%d stations, columns: %s' % (len(stations_table), list(stations_table.columns)))

events, assignments = associator.associate(picks, stations_table)

associator.transform_events(events)
events['time'] = events['time'].apply(UTCDateTime)
all_pick_assignments = pd.merge(events, assignments, left_on="idx", right_on="event_idx", suffixes=("", "_pick"))

events.to_csv(out_path + 'events_detect_octo_staonly.csv', index=False)
all_pick_assignments.to_csv(out_path + 'all_pick_assignments_staonly.csv', index=False)

29 stations, columns: ['id', 'longitude', 'latitude', 'elevation', 'x', 'y', 'z']


In [11]:
all_pick_assignments

,idx,time,x,y,z,picks,latitude,longitude,depth,event_idx,pick_idx,residual,station,phase,time_pick
0,0,2023-08-23T22:12:51.923964Z,-18.941724,39.120686,5.625,13,46.851648,-121.748363,5.625,0,58076,0.545373,ARAT,S,1.692829e+09
1,0,2023-08-23T22:12:51.923964Z,-18.941724,39.120686,5.625,13,46.851648,-121.748363,5.625,0,4,-0.444375,COPP,P,1.692829e+09
2,0,2023-08-23T22:12:51.923964Z,-18.941724,39.120686,5.625,13,46.851648,-121.748363,5.625,0,58080,0.254565,COPP,S,1.692829e+09
3,0,2023-08-23T22:12:51.923964Z,-18.941724,39.120686,5.625,13,46.851648,-121.748363,5.625,0,55783,1.094198,FMW,P,1.692829e+09
4,0,2023-08-23T22:12:51.923964Z,-18.941724,39.120686,5.625,13,46.851648,-121.748363,5.625,0,181634,-0.824679,FMW,S,1.692829e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32518,1494,2023-11-21T22:36:19.014443Z,-59.597130,-74.581594,14.375,16,45.826454,-122.266966,14.375,1494,55729,0.085747,PARA,P,1.700606e+09
32519,1494,2023-11-21T22:36:19.014443Z,-59.597130,-74.581594,14.375,16,45.826454,-122.266966,14.375,1494,181588,-0.131379,PARA,S,1.700606e+09
32520,1494,2023-11-21T22:36:19.014443Z,-59.597130,-74.581594,14.375,16,45.826454,-122.266966,14.375,1494,55740,-0.567082,PR05,P,1.700606e+09
32521,1494,2023-11-21T22:36:19.014443Z,-59.597130,-74.581594,14.375,16,45.826454,-122.266966,14.375,1494,55767,0.018880,SIFT,P,1.700606e+09


In [12]:
events

,idx,time,x,y,z,picks,latitude,longitude,depth
0,0,2023-08-23T22:12:51.923964Z,-18.941724,39.120686,5.625,13,46.851648,-121.748363,5.625
1,1,2023-08-24T00:47:35.705801Z,-18.017737,37.813763,0.625,14,46.839917,-121.736196,0.625
2,2,2023-08-24T01:55:13.612553Z,-18.941724,43.041454,0.625,12,46.886916,-121.748526,0.625
3,3,2023-08-24T02:58:40.362326Z,-19.865710,40.427609,4.375,10,46.863377,-121.760535,4.375
4,4,2023-08-24T03:15:05.196174Z,-18.017737,37.813763,0.625,18,46.839917,-121.736196,0.625
...,...,...,...,...,...,...,...,...,...
1490,1490,2023-11-21T05:02:04.845693Z,-18.017737,39.120686,0.625,13,46.851673,-121.736248,0.625
1491,1491,2023-11-21T07:55:08.155085Z,-14.321791,37.813763,5.625,14,46.840007,-121.687746,5.625
1492,1492,2023-11-21T21:29:47.259627Z,105.796456,167.199117,0.625,15,47.995170,-120.082416,0.625
1493,1493,2023-11-21T21:36:52.300769Z,-117.808281,-87.650822,3.125,27,45.701427,-123.012715,3.125


### strict parameters - 5 station

In [13]:
associator = pyocto.OctoAssociator.from_area(
    lat=(45, 48),
    lon=(-123, -120),
    zlim=(0, 40),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=6,
    n_p_picks=3,
    n_s_picks=3,
    n_p_and_s_picks=3,
)


# CC-KAUT, UW-LON, CC-MILD, CC-PARA, UW-RER
small_stations_table = stations_table[stations_table['id'].isin(['KAUT', 'LON', 'MILD', 'PARA', 'RER'])]
small_stations_table 

,id,longitude,latitude,elevation,x,y,z
6,KAUT,-121.857381,46.730263,689.00,-27.317449,25.658837,-0.68900
7,MILD,-121.775680,46.808100,2015.59,-21.042066,34.286563,-2.01559
11,PARA,-121.742195,46.786441,1651.18,-18.493647,31.870380,-1.65118
25,LON,-121.809601,46.750599,853.00,-23.656352,27.904023,-0.85300
27,RER,-121.842500,46.818580,1751.10,-26.137212,35.471654,-1.75110


In [14]:
# Association 
# Re-project the stations here rather than relying on the table cell having been run in this
# kernel: associate() hard-errors if x/y/z are missing, and the table is rebuilt from the two
# inventories on every run so it arrives with lon/lat/elev only. transform_stations is
# idempotent -- it recomputes x/y/z from latitude/longitude/elevation -- so calling it in both
# places costs nothing.
associator.transform_stations(small_stations_table)
print('%d stations, columns: %s' % (len(small_stations_table), list(small_stations_table.columns)))

events, assignments = associator.associate(picks, small_stations_table)

associator.transform_events(events)
events['time'] = events['time'].apply(UTCDateTime)
all_pick_assignments = pd.merge(events, assignments, left_on="idx", right_on="event_idx", suffixes=("", "_pick"))

events.to_csv(out_path + 'events_detect_octo_staonly_only_KAUT-LON-MILK-PARA-RER-11032026_strict.csv', index=False)
all_pick_assignments.to_csv(out_path + 'all_pick_assignments_staonly_only_KAUT-LON-MILK-PARA-RER-11032026_strict.csv', index=False)

5 stations, columns: ['id', 'longitude', 'latitude', 'elevation', 'x', 'y', 'z']
Found stations with picks but without station information: ARAT CARB COPP CRBN FMW GNOB GTWY OBSR OPCH PANH PR01 PR02 PR03 PR04 PR05 RCM RUSH SIFT STYX TABR TAVI VOIT WOW WPEQ


In [15]:
all_pick_assignments

,idx,time,x,y,z,picks,latitude,longitude,depth,event_idx,pick_idx,residual,station,phase,time_pick
0,0,2023-08-24T06:56:25.035232Z,-19.865710,40.427609,0.625,10,46.863377,-121.760535,0.625,0,170,0.509943,KAUT,P,1.692860e+09
1,0,2023-08-24T06:56:25.035232Z,-19.865710,40.427609,0.625,10,46.863377,-121.760535,0.625,0,58358,0.337277,KAUT,S,1.692860e+09
2,0,2023-08-24T06:56:25.035232Z,-19.865710,40.427609,0.625,10,46.863377,-121.760535,0.625,0,55879,0.012102,LON,P,1.692860e+09
3,0,2023-08-24T06:56:25.035232Z,-19.865710,40.427609,0.625,10,46.863377,-121.760535,0.625,0,181859,-0.111820,LON,S,1.692860e+09
4,0,2023-08-24T06:56:25.035232Z,-19.865710,40.427609,0.625,10,46.863377,-121.760535,0.625,0,195,-0.222802,MILD,P,1.692860e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2829,353,2023-11-21T01:56:28.407647Z,-31.877535,31.279149,10.625,9,46.780618,-121.917427,10.625,353,181307,0.064687,MILD,S,1.700532e+09
2830,353,2023-11-21T01:56:28.407647Z,-31.877535,31.279149,10.625,9,46.780618,-121.917427,10.625,353,55715,-0.014954,PARA,P,1.700532e+09
2831,353,2023-11-21T01:56:28.407647Z,-31.877535,31.279149,10.625,9,46.780618,-121.917427,10.625,353,181556,0.115838,PARA,S,1.700532e+09
2832,353,2023-11-21T01:56:28.407647Z,-31.877535,31.279149,10.625,9,46.780618,-121.917427,10.625,353,58057,0.041069,RER,P,1.700532e+09


In [16]:
events

,idx,time,x,y,z,picks,latitude,longitude,depth
0,0,2023-08-24T06:56:25.035232Z,-19.865710,40.427609,0.625,10,46.863377,-121.760535,0.625
1,1,2023-08-24T15:24:01.184268Z,-21.713683,36.506840,10.625,8,46.828051,-121.784584,10.625
2,2,2023-08-25T06:01:38.711731Z,-32.801521,53.496836,11.875,10,46.980423,-121.931124,11.875
3,3,2023-08-25T07:23:15.797731Z,-32.801521,53.496836,11.875,10,46.980423,-121.931124,11.875
4,4,2023-08-26T07:06:46.703143Z,-23.561656,45.655300,0.625,8,46.910280,-121.809276,0.625
...,...,...,...,...,...,...,...,...,...
349,349,2023-11-20T13:57:45.661564Z,-24.485643,32.586072,3.125,7,46.792686,-121.820704,3.125
350,350,2023-11-20T18:59:18.118991Z,6.929899,-23.611607,33.125,10,46.287552,-121.410073,33.125
351,351,2023-11-20T20:42:43.971611Z,-34.649494,41.734532,8.125,9,46.874527,-121.954516,8.125
352,352,2023-11-20T22:15:45.017741Z,-43.889359,67.872987,11.875,10,47.109090,-122.078245,11.875
